###  Importing dependencies

In [4]:
import cv2
import numpy as np
import requests
import threading
import time
from ultralytics import YOLO
%matplotlib inline

### CONFIGURATION — edit these to match your setup

In [5]:
STREAM_URL     = "http://192.168.137.75:4747/videofeed"  # DroidCam stream URL
MODEL_PATH     = "yolov5su.pt"   # YOLO model file
CONFIDENCE     = 0.4             # Detection confidence threshold (0.0 - 1.0)
FRAME_WIDTH    = 640             # Display/recording frame width
FRAME_HEIGHT   = 480             # Display/recording frame height
FPS            = 20              # Recording frames per second



####  STEP 1 — Load the YOLO model

In [6]:
print("🤖 Loading YOLO model...")
model = YOLO(MODEL_PATH)
print("✅ Model loaded.")

🤖 Loading YOLO model...
✅ Model loaded.


#### STEP 2 — Background thread to read the live stream

In [7]:
# Why a separate thread?
#   YOLO inference takes time. Without threading, frames pile up
#   in the buffer and cause lag. This thread continuously reads
#   the latest frame so the main thread always gets fresh video.
# --------------------------------------------------------------
latest_frame = None          # Shared variable holding the newest frame
lock = threading.Lock()      # Prevents simultaneous read/write conflicts


def stream_reader():
    """
    Continuously reads MJPEG frames from the HTTP stream.
    Stores only the latest frame in `latest_frame`.
    MJPEG frames are JPEG images delimited by FF D8 (start) and FF D9 (end).
    """
    global latest_frame

    stream = requests.get(STREAM_URL, stream=True, timeout=10)
    bytes_buffer = b''  # accumulate raw bytes from the stream

    for chunk in stream.iter_content(chunk_size=4096):
        bytes_buffer += chunk

        # Find JPEG start and end markers in the byte buffer
        start = bytes_buffer.find(b'\xff\xd8')  # JPEG start marker
        end   = bytes_buffer.find(b'\xff\xd9')  # JPEG end marker

        if start != -1 and end != -1:
            # Extract one complete JPEG frame
            jpg_data = bytes_buffer[start:end + 2]

            # Discard everything up to and including this frame
            bytes_buffer = bytes_buffer[end + 2:]

            # Decode JPEG bytes into an OpenCV image (numpy array)
            np_arr = np.frombuffer(jpg_data, dtype=np.uint8)
            frame  = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

            if frame is not None:
                with lock:
                    latest_frame = frame  # always keep only the newest frame


# Start the stream reader in a background daemon thread
# (daemon=True means it auto-stops when the main program exits)
stream_thread = threading.Thread(target=stream_reader, daemon=True)
stream_thread.start()


#### STEP 3 — Wait until the first frame arrives

In [8]:
print("⏳ Waiting for stream...")
while True:
    with lock:
        if latest_frame is not None:
            break
    time.sleep(0.1)

print("✅ Stream connected! Press 'r' to record, 's' for snapshot, 'q' to quit.")

⏳ Waiting for stream...
✅ Stream connected! Press 'r' to record, 's' for snapshot, 'q' to quit.


#### STEP 4 — Set up video recording

In [9]:
recording = False   # recording state flag
writer    = None    # VideoWriter object (created when recording starts)
out_path  = None    # output file path


def start_recording():
    """Create a VideoWriter and begin saving frames."""
    global writer, out_path, recording
    out_path = f"recording_{int(time.time())}.mp4"
    fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
    writer   = cv2.VideoWriter(out_path, fourcc, FPS, (FRAME_WIDTH, FRAME_HEIGHT))
    recording = True
    print(f"🔴 Recording started → {out_path}")


def stop_recording():
    """Release the VideoWriter and stop saving frames."""
    global writer, recording
    recording = False
    writer.release()
    writer = None
    print(f"⏹️  Recording saved → {out_path}")

#### STEP 5 — Main detection loop

In [13]:
prev_time = time.time()  # used to calculate FPS

while True:

    # --- Grab the latest frame safely ---
    with lock:
        if latest_frame is None:
            continue
        frame = latest_frame.copy()

    # --- Resize for consistent processing ---
    frame = cv2.resize(frame, (FRAME_WIDTH, FRAME_HEIGHT))

    # --- Run YOLO object detection ---
    results        = model(frame, conf=CONFIDENCE, verbose=False)
    annotated_frame = results[0].plot()  # draw bounding boxes on frame

    # --- Calculate real-time FPS ---
    curr_time = time.time()
    fps       = 1 / (curr_time - prev_time + 1e-6)
    prev_time = curr_time

    # --- Overlay: FPS and object count ---
    cv2.putText(annotated_frame, f"FPS: {fps:.1f}",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(annotated_frame, f"Objects: {len(results[0].boxes)}",
                (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # --- Overlay: recording indicator ---
    if recording:
        cv2.circle(annotated_frame, (FRAME_WIDTH - 20, 20), 10, (0, 0, 255), -1)
        cv2.putText(annotated_frame, "REC",
                    (FRAME_WIDTH - 55, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        writer.write(annotated_frame)  # save this frame to video file

    # --- Show the frame ---
    cv2.imshow("YOLO Real-Time Detector", annotated_frame)

    # --- Keyboard controls ---
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        print("👋 Quitting...")
        break

    elif key == ord('r'):
        if not recording:
            start_recording()
        else:
            stop_recording()

    elif key == ord('s'):
        snap_path = f"snapshot_{int(time.time())}.jpg"
        cv2.imwrite(snap_path, annotated_frame)
        print(f"📸 Snapshot saved → {snap_path}")

        
#STEP 6 — Cleanup
if recording and writer:
    stop_recording()  # save video if still recording when quitting

cv2.destroyAllWindows()
print("✅ Done.")




#   Real-Time Object Detector using YOLOv5 + DroidCam Stream
#   Controls:  'r' = start/stop recording
#              's' = save snapshot
#              'q' = quit


👋 Quitting...
✅ Done.
